# Ingestion : scraper les comptes annuels sur la Centrale des bilans (NBB/CBSO)

Objectif : à partir des numéros d'entreprise déjà en base, aller chercher les comptes annuels publiés sur le site de la Centrale des bilans de la Banque Nationale de Belgique

## 1. Découverte du site

Avant d'écrire la moindre ligne de code, allez consulter le site normalement, dans un navigateur : https://consult.cbso.nbb.be/consult-enterprise/0693810613

C'est la fiche d'une entreprise réelle, remplacez le numéro par n'importe quel numéro BCE (10 chiffres, sans points) pour voir une autre entreprise. Vous devriez voir la liste des comptes annuels déposés, année par année.

**À noter avant de continuer :**

- Depuis l'exercice comptable **2021**, les comptes sont disponibles au format **CSV** (en plus du PDF). Avant 2021, seul le PDF existe. C'est pourquoi on se concentre sur le CSV et sur les années récentes dans cet exercice, le PDF reste possible à télécharger en plus si vous voulez aller plus loin.(OCR)
- Cette page HTML n'est pas elle-même la source de données, elle appelle une API JSON, que vous allez interroger directement dans la suite.

## 2. Premier appel : lister les dépôts d'une entreprise

**Endpoint** : `https://consult.cbso.nbb.be/api/rs-consult/published-deposits`

**Paramètres de requête (query params)** à envoyer :

- `enterpriseNumber` : le numéro BCE, SANS points (ex. `0693810613`, pas `0693.810.613`)
- `page` : numéro de page, en partant de `0`
- `size` : taille de page (ex. `50`)
- `sort` : à envoyer comme une liste, avec DEUX valeurs (`periodEndDate,desc` et `depositDate,desc`) -- avec la librairie `requests`, un paramètre dont la valeur est une liste Python est automatiquement répété deux fois dans l'URL, ce qui correspond au format attendu par cette API

**Pagination** : la réponse JSON contient un champ `content` (la liste des dépôts de cette page) et un champ booléen `last`. Continuez à demander la page suivante tant que `last` vaut `false`. Attention : il n'y a PAS de champ `totalPages` malgré ce qu'on pourrait attendre d'une API paginée classique !!!

**En-têtes (headers)** à envoyer:

- `User-Agent` : une chaîne de navigateur réaliste (pas du n'importe quoi)
- `Accept`, `Accept-Language` (c'est mieux si c'est en francais)
- `Referer` : l'URL de la fiche entreprise correspondante (`https://consult.cbso.nbb.be/consult-enterprise/{numero}`)

**Session/cookies** : avant d'appeler l'API, faites d'abord une requête GET normale vers la fiche HTML de l'entreprise (`consult-enterprise/{numero}`) pour récupérer les cookies que le site pose sur un chargement de page classique. Réutilisez ensuite ces cookies pour l'appel API 


Chaque dépôt renvoyé contient au moins : `id` (identifiant du dépôt), `periodEndDateYear`, `language`, `modelName`

In [17]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [18]:
from __future__ import annotations

import os
import time
from pprint import pprint

import requests

BASE = "https://consult.cbso.nbb.be"
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


def _session_for(enterprise_number: str) -> requests.Session:
    """Ouvre une session avec les en-têtes attendus, et pose d'abord les cookies
    en chargeant la fiche HTML de l'entreprise avant tout appel API (comme le
    ferait un navigateur normal)."""
    session = requests.Session()
    session.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "fr-FR,fr;q=0.9",
    })
    fiche_url = f"{BASE}/consult-enterprise/{enterprise_number}"
    session.get(fiche_url, timeout=15)  # pose les cookies ; le contenu HTML ne nous intéresse pas
    session.headers["Referer"] = fiche_url
    return session


def list_deposits(enterprise_number: str, session: requests.Session | None = None, page_size: int = 50) -> list[dict]:
    """Liste tous les dépôts publiés d'une entreprise (numéro BCE SANS points),
    toutes pages confondues. Pas de champ `totalPages` : on avance tant que
    `last` est `false`."""
    session = session or _session_for(enterprise_number)
    session.headers["Accept"] = "application/json"

    deposits = []
    page = 0
    while True:
        resp = session.get(
            f"{BASE}/api/rs-consult/published-deposits",
            params={
                "enterpriseNumber": enterprise_number,
                "page": page,
                "size": page_size,
                # une valeur de liste => requests répète le paramètre deux fois dans l'URL
                "sort": ["periodEndDate,desc", "depositDate,desc"],
            },
            timeout=15,
        )
        resp.raise_for_status()
        data = resp.json()
        deposits.extend(data["content"])
        if data["last"]:
            break
        page += 1
    return deposits


sample_deposits = list_deposits("0693810613")
print(f"{len(sample_deposits)} dépôt(s) trouvé(s)")
for d in sample_deposits[:3]:
    print(d["id"], d["periodEndDateYear"], d["language"], d["modelName"])


8 dépôt(s) trouvé(s)
3cf4404a-7ba0-11f1-92d9-1db02102d1ba 2025 NL Verkort model kapitaalloze vennootschap
4f64a96c-58e4-11f0-86e3-459f89c72080 2024 NL Verkort model kapitaalloze vennootschap
b34a708c-3df0-11ef-bb31-d5454915068a 2023 NL Verkort model kapitaalloze vennootschap


## 3. Télécharger un dépôt CSV

Une fois qu'on a l'`id` d'un dépôt (récupéré à l'étape précédente), le CSV se télécharge via :

`https://consult.cbso.nbb.be/api/external/broker/public/deposits/consult/csv/{id}`

Exemple concret (id réel) : https://consult.cbso.nbb.be/api/external/broker/public/deposits/consult/csv/3cf4404a-7ba0-11f1-92d9-1db02102d1ba

Points à gérer :

- Réutilisez la même session (mêmes cookies/en-têtes que pour l'étape 2).
- Un statut `404` ou `500` veut dire qu'il n'y a pas de CSV pour ce dépôt (normal, ne pas relancer)
- Un statut `502`/`503` est temporaire (le serveur a un souci passager), celui-là, il faut le réessayer plus tard, pas l'ignorer. (timeout ou bien skip)
- Une réponse `200` mais avec un contenu très court (quelques dizaines d'octets) correspond en général à un fichier vide, à traiter comme "pas de fichier" aussi.

In [19]:
def download_csv(deposit_id: str, session: requests.Session, max_retries: int = 3) -> bytes | None:
    """Télécharge le CSV d'un dépôt donné.

    Renvoie None si le dépôt n'a pas de CSV : 404/500 (pas de fichier pour ce
    dépôt, ne pas réessayer), ou 200 avec un contenu quasi vide (quelques
    dizaines d'octets -> fichier vide en pratique).
    Réessaie sur 502/503 (souci serveur passager) jusqu'à `max_retries` fois,
    avec un petit délai croissant entre les tentatives.
    """
    url = f"{BASE}/api/external/broker/public/deposits/consult/csv/{deposit_id}"

    for attempt in range(max_retries):
        resp = session.get(url, timeout=30)

        if resp.status_code in (404, 500):
            return None
        if resp.status_code in (502, 503):
            time.sleep(5 * (attempt + 1))
            continue

        resp.raise_for_status()

        if len(resp.content) < 100:  # 200 mais contenu quasi vide -> pas de fichier réel
            return None
        return resp.content

    return None  # 502/503 persistant au-delà de max_retries : on laisse tomber pour cette fois


## 4. Scraper en continu jusqu'au 429, et gérer le cooldown

Faites tourner vos appels en boucle sur plusieurs entreprises, jusqu'à obtenir un code `429 Too Many Requests`.

quand vous obtenez ce 429, affichez l'intégralité des en-têtes de la réponse (`dict(resp.headers)`). Certaines API renvoient un en-tête `Retry-After` qui indique précisément combien de secondes attendre avant de réessayer, c'est à vous de regarder si CBSO l'envoie.

- **Si l'en-tête est présent** : attendez exactement la durée qu'il indique avant de réessayer.
- **Si l'en-tête est absent** : repli sur un backoff exponentiel (attendre un peu, puis de plus en plus longtemps à chaque 429 consécutif, jusqu'à un plafond raisonnable comme 120 secondes)

In [20]:
from __future__ import annotations

import os
import time
from pprint import pprint

import requests

BASE = "https://consult.cbso.nbb.be"
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


def _session_for(enterprise_number: str, max_network_retries: int = 5) -> requests.Session:
    """Ouvre une session avec les en-têtes attendus, et pose d'abord les cookies
    en chargeant la fiche HTML de l'entreprise avant tout appel API (comme le
    ferait un navigateur normal). Réessaie en cas de souci réseau (timeout,
    connexion refusée, etc.) avant d'abandonner."""
    session = requests.Session()
    session.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "fr-FR,fr;q=0.9",
    })
    fiche_url = f"{BASE}/consult-enterprise/{enterprise_number}"

    backoff = 1
    for attempt in range(max_network_retries):
        try:
            session.get(fiche_url, timeout=30)  # pose les cookies ; le contenu HTML ne nous intéresse pas
            break
        except requests.exceptions.RequestException as exc:
            if attempt == max_network_retries - 1:
                raise
            print(f"Erreur réseau sur la fiche {enterprise_number} ({exc.__class__.__name__}), "
                  f"retry dans {backoff:.0f}s...")
            time.sleep(backoff)
            backoff = min(backoff * 2, 60)

    session.headers["Referer"] = fiche_url
    return session


def list_deposits(enterprise_number: str, session: requests.Session | None = None, page_size: int = 50) -> list[dict]:
    """Liste tous les dépôts publiés d'une entreprise (numéro BCE SANS points),
    toutes pages confondues. Pas de champ `totalPages` : on avance tant que
    `last` est `false`."""
    session = session or _session_for(enterprise_number)
    session.headers["Accept"] = "application/json"

    deposits = []
    page = 0
    while True:
        resp = session.get(
            f"{BASE}/api/rs-consult/published-deposits",
            params={
                "enterpriseNumber": enterprise_number,
                "page": page,
                "size": page_size,
                "sort": ["periodEndDate,desc", "depositDate,desc"],
            },
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        deposits.extend(data["content"])
        if data["last"]:
            break
        page += 1
    return deposits


sample_deposits = list_deposits("0693810613")
print(f"{len(sample_deposits)} dépôt(s) trouvé(s)")
for d in sample_deposits[:3]:
    print(d["id"], d["periodEndDateYear"], d["language"], d["modelName"])

8 dépôt(s) trouvé(s)
3cf4404a-7ba0-11f1-92d9-1db02102d1ba 2025 NL Verkort model kapitaalloze vennootschap
4f64a96c-58e4-11f0-86e3-459f89c72080 2024 NL Verkort model kapitaalloze vennootschap
b34a708c-3df0-11ef-bb31-d5454915068a 2023 NL Verkort model kapitaalloze vennootschap


## 5. Stocker les fichiers dans HDFS + suivi scrapping

**Stockage HDFS** : un dossier par entreprise, avec les CSV dedans, respectez la structure :

`/data/raw/{numero_entreprise}/cbso/csvs/{annee}.csv`

Avant d'écrire un fichier, vérifiez s'il existe déjà à ce chemin (pour ne pas retélécharger ce qu'on a déjà).

**Suivi des entreprises déjà traitées** : en plus de la vérification par fichier HDFS ci-dessus, tenez un petit fichier JSON simple qui note, pour chaque entreprise déjà passée en revue, si elle a été traitée, sauter directement les entreprises déjà vues

In [21]:
pip install hdfs

Note: you may need to restart the kernel to use updated packages.


In [22]:
# pip install hdfs   (client WebHDFS pur Python)
import json
from pathlib import Path

from hdfs import InsecureClient

HDFS_URL = os.environ.get("HDFS_URL", "http://localhost:9870")  # endpoint WebHDFS (namenode)
hdfs_client = InsecureClient(HDFS_URL)

TRACKING_FILE = Path("scraping_seen.json")  # petit fichier JSON local de suivi


def _load_tracking() -> dict:
    if TRACKING_FILE.exists():
        return json.loads(TRACKING_FILE.read_text())
    return {}


def _save_tracking(tracking: dict) -> None:
    TRACKING_FILE.write_text(json.dumps(tracking, indent=2))


def hdfs_path_for(enterprise_number: str, year: int) -> str:
    return f"/data/raw/{enterprise_number}/cbso/csvs/{year}.csv"


def scrape_enterprise(enterprise_number: str) -> int:
    """Scrape une entreprise : liste ses dépôts (section 4), télécharge (section 3)
    et stocke sur HDFS chaque CSV manquant. Renvoie le nombre de fichiers
    effectivement téléchargés cette fois-ci.

    Sauté d'emblée si l'entreprise est déjà marquée "traitée" dans le fichier
    de suivi JSON local ; sinon, vérifie fichier par fichier sur HDFS avant de
    retélécharger quoi que ce soit.
    """
    tracking = _load_tracking()
    if tracking.get(enterprise_number, {}).get("done"):
        return 0

    session = _session_for(enterprise_number)
    deposits = list_deposits_safe(enterprise_number, session=session)

    downloaded = 0
    for deposit in deposits:
        year = deposit["periodEndDateYear"]
        path = hdfs_path_for(enterprise_number, year)

        if hdfs_client.status(path, strict=False) is not None:
            continue  # déjà présent sur HDFS, on ne retélécharge pas

        content = download_csv(deposit["id"], session)
        if content is None:
            continue

        hdfs_client.write(path, data=content, overwrite=False)
        downloaded += 1

    tracking[enterprise_number] = {"done": True, "documentsDownloaded": downloaded}
    _save_tracking(tracking)
    return downloaded


## 6. Passer par Tor : un premier exemple simple

**Service Docker** (`docker-compose.yml`) : un conteneur Tor avec l'image `dperson/torproxy` expose un proxy SOCKS5 sur le port `9050`, par exemple :

```yaml
  tor1:
    image: dperson/torproxy
    ports:
      - "9050:9050"
      - "9051:9051"
```

**Requête Python via Tor** : la librairie `requests` sait parler à un proxy SOCKS5 nativement, à condition d'installer `requests[socks]` (qui installe PySocks). Il suffit de configurer `session.proxies` avec une URL au format `socks5h://<hôte>:9050` (le `h` final est important : il dit à `requests` de résoudre les noms de domaine À TRAVERS Tor aussi, pas seulement le trafic).

Faites un test simple : une requête GET vers un service qui renvoie votre IP publique (par exemple un endpoint "what is my ip"), une fois SANS proxy, une fois AVEC le proxy Tor -- vous devriez voir deux IP différentes, ce qui confirme que le trafic passe bien par Tor.

In [8]:
pip install "requests[socks]"

Note: you may need to restart the kernel to use updated packages.


In [13]:
# pip install "requests[socks]"
IP_CHECK_URL = "https://api.ipify.org?format=json"

resp_direct = requests.get(IP_CHECK_URL, timeout=15)
print("Sans proxy :", resp_direct.json())

tor_session = requests.Session()
tor_session.proxies = {
    "http": "socks5h://localhost:9050",   # le "h" final : résolution DNS via Tor aussi
    "https": "socks5h://localhost:9050",
}
resp_tor = tor_session.get(IP_CHECK_URL, timeout=60)
print("Avec proxy Tor :", resp_tor.json())


Sans proxy : {'ip': '82.96.161.255'}
Avec proxy Tor : {'ip': '45.91.250.107'}


## 7. Plusieurs instances Tor, avec rotation

on ne veut pas exposer notre IP réelle, pour pouvoir continuer à utiliser le site à des fins de recherche sans risquer un blocage définitif de notre propre adresse. Faire tourner le trafic sur plusieurs identités Tor, et changer d'identité quand l'une d'elles se fait bloquer/limiter, permet de continuer à travailler sans jamais exposer la vraie IP de la machine.

créez 3 services Tor distincts dans le docker-compose (même image que l'étape 6, des noms différents, par ex. `tor1`/`tor2`/`tor3`, des ports différents sur l'hôte si vous voulez y accéder depuis votre machine, mais en interne au réseau docker, chacun écoute toujours sur 9050/9051).

Tor expose un "port de contrôle" (9051 par défaut) qui accepte une commande `SIGNAL NEWNYM` pour forcer la construction d'un nouveau circuit (donc une nouvelle IP de sortie) sur demande. Ce port demande une authentification (mot de passe)

Écrivez une petite classe ou fonction qui : garde une liste de vos 3 proxies, envoie les requêtes via le proxy courant, et sur un 429 (ou un blocage), envoie `SIGNAL NEWNYM` au proxy courant PUIS passe au proxy suivant de la liste avant de réessayer.

# Section 8 (robust_get + list_deposits_safe)

In [23]:
def robust_get(session: requests.Session, url: str, max_backoff: int = 120,
                max_network_retries: int = 5, **kwargs) -> requests.Response:
    """GET qui absorbe à la fois :
    - les 429 : respecte l'en-tête Retry-After s'il est présent, sinon
      backoff exponentiel plafonné à `max_backoff` secondes.
    - les erreurs réseau (timeout, connexion coupée, DNS, etc.) : retry avec
      backoff exponentiel, jusqu'à `max_network_retries` tentatives, puis
      abandonne en laissant remonter l'exception."""
    backoff = 1
    network_attempts = 0

    while True:
        try:
            resp = session.get(url, **kwargs)
        except requests.exceptions.RequestException as exc:
            network_attempts += 1
            if network_attempts > max_network_retries:
                print(f"Abandon après {max_network_retries} erreurs réseau sur {url}")
                raise
            wait = min(backoff, max_backoff)
            print(f"Erreur réseau ({exc.__class__.__name__}), attente {wait:.0f}s avant retry "
                  f"({network_attempts}/{max_network_retries})...")
            time.sleep(wait)
            backoff = min(backoff * 2, max_backoff)
            continue

        if resp.status_code != 429:
            return resp

        print("429 reçu, en-têtes de la réponse :")
        pprint(dict(resp.headers))

        retry_after = resp.headers.get("Retry-After")
        if retry_after:
            wait = float(retry_after)
        else:
            wait = backoff
            backoff = min(backoff * 2, max_backoff)

        print(f"Attente de {wait:.0f}s avant de réessayer...")
        time.sleep(wait)


def list_deposits_safe(enterprise_number: str, session: requests.Session | None = None, page_size: int = 50) -> list[dict]:
    """Comme list_deposits, mais chaque appel passe par robust_get : un 429 ou
    une erreur réseau est absorbé (attente puis retry) plutôt que de remonter
    telle quelle."""
    session = session or _session_for(enterprise_number)
    deposits = []
    page = 0
    while True:
        resp = robust_get(
            session,
            f"{BASE}/api/rs-consult/published-deposits",
            params={
                "enterpriseNumber": enterprise_number,
                "page": page,
                "size": page_size,
                "sort": ["periodEndDate,desc", "depositDate,desc"],
            },
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        deposits.extend(data["content"])
        if data["last"]:
            break
        page += 1
    return deposits

## 9. Cibler intelligemment : échantillonnage stratifié par forme juridique

1. Récupérez, depuis MongoDB, la liste des valeurs distinctes du champ `JuridicalForm` présentes dans la collection `entreprise`.
2. Pour chaque valeur, tirez un échantillon (n>100) ALÉATOIRE d'une centaine d'entreprises ayant cette forme juridique 
3. Pour chaque entreprise de l'échantillon, faites juste l'appel de listage
4. Calculez, par forme juridique, le pourcentage d'entreprises SANS aucun dépôt.
5. Les formes juridiques dont ce pourcentage dépasse **95%** 

Le résultat de cette étape est une LISTE DE FORMES JURIDIQUES À EXCLURE, que vous réutiliserez à l'étape suivante pour filtrer les entreprises à traiter réellement.

In [ ]:
import pymongo

mongo_client = pymongo.MongoClient(os.environ.get("MONGO_URI", "mongodb://localhost:27017"))
db = mongo_client[os.environ.get("MONGO_DB", "kbo")]

SAMPLE_SIZE = 120  # > 100 demandé, marge pour couvrir d'éventuels échecs réseau

juridical_forms = db.enterprise.distinct("JuridicalForm")
print(f"{len(juridical_forms)} formes juridiques distinctes")


def sample_enterprise_numbers(juridical_form: str, sample_size: int = SAMPLE_SIZE) -> list[str]:
    """Tire un échantillon ALÉATOIRE d'EnterpriseNumber pour une forme juridique
    donnée, via $sample (échantillonnage aléatoire côté MongoDB, pas en mémoire)."""
    pipeline = [
        {"$match": {"JuridicalForm": juridical_form}},
        {"$sample": {"size": sample_size}},
        {"$project": {"_id": 0, "EnterpriseNumber": 1}},
    ]
    return [doc["EnterpriseNumber"] for doc in db.enterprise.aggregate(pipeline)]


def deposit_coverage(juridical_form: str) -> float:
    """Pour un échantillon de cette forme juridique, fait juste l'appel de
    listage des dépôts (pas de téléchargement CSV ici) et renvoie le
    pourcentage d'entreprises SANS aucun dépôt trouvé.
    Une entreprise dont l'appel échoue définitivement (erreur réseau
    persistante) est comptée à part, pas mêlée aux 'sans dépôt'."""
    sample = sample_enterprise_numbers(juridical_form)
    without_deposits = 0
    failed = 0
    for enterprise_number in sample:
        clean_number = enterprise_number.replace(".", "")  # l'API veut le numéro SANS points
        try:
            if not list_deposits_safe(clean_number):
                without_deposits += 1
        except requests.exceptions.RequestException as exc:
            failed += 1
            print(f"Échec définitif sur {clean_number} ({exc.__class__.__name__}), ignoré du calcul")

    usable = len(sample) - failed
    if usable == 0:
        return float("nan")
    return 100 * without_deposits / usable


coverage_by_form = {form: deposit_coverage(form) for form in juridical_forms}
for form, pct in sorted(coverage_by_form.items(), key=lambda kv: -kv[1]):
    print(f"{form:60s} {pct:5.1f}% sans dépôt")

EXCLUDED_JURIDICAL_FORMS = [form for form, pct in coverage_by_form.items() if pct > 95]
print("\nFormes juridiques exclues (>95% sans dépôt) :")
pprint(EXCLUDED_JURIDICAL_FORMS)


103 formes juridiques distinctes
429 reçu, en-têtes de la réponse :
{'Cache-Control': 'no-store',
 'Connection': 'close',
 'Content-Length': '1484',
 'Content-Type': 'text/html',
 'Date': 'Tue, 28 Jul 2026 15:27:26 GMT',
 'X-Cache': 'CONFIG_NOCACHE',
 'x-azure-ref': '20260728T152726Z-r15744c59f4bqhsvhC1FRA65rw000000044g00000000ahnz'}
Attente de 1s avant de réessayer...
429 reçu, en-têtes de la réponse :
{'Cache-Control': 'no-store',
 'Connection': 'close',
 'Content-Length': '1484',
 'Content-Type': 'text/html',
 'Date': 'Tue, 28 Jul 2026 15:27:29 GMT',
 'X-Cache': 'CONFIG_NOCACHE',
 'x-azure-ref': '20260728T152729Z-r15744c59f46cz7lhC1FRA72e400000006gg00000000kcty'}
Attente de 1s avant de réessayer...
429 reçu, en-têtes de la réponse :
{'Cache-Control': 'no-store',
 'Connection': 'close',
 'Content-Length': '1484',
 'Content-Type': 'text/html',
 'Date': 'Tue, 28 Jul 2026 15:27:30 GMT',
 'X-Cache': 'CONFIG_NOCACHE',
 'x-azure-ref': '20260728T152730Z-r15744c59f4pcx2chC1FRAdauc00000000xg0

## 10. Collection MongoDB de suivi du scraping

Créez une nouvelle collection, avec un document par entreprise CIBLE

- `enterpriseNumber`
- `juridicalForm` (pour pouvoir vérifier après coup que le filtre a bien été appliqué)
- `status` (`pending`, `done`, `error`)
- `lastScrapedAt` (timestamp du dernier passage)
- `documentsDownloaded` (nombre de CSV effectivement récupérés pour cette entreprise)

ne traiter que les entreprises dont la forme juridique n'est pas exclue ET dont le statut n'est pas déjà `done`. À chaque entreprise traitée, mettez à jour son document dans cette collection.

In [ ]:
from datetime import datetime, timezone


def init_scraping_tracking(excluded_forms: list[str]) -> None:
    """Crée un document 'pending' pour chaque entreprise CIBLE (forme juridique
    non exclue), sans écraser celles déjà présentes (utile si on relance ce
    notebook plusieurs fois)."""
    db.scraping_tracking.create_index("enterpriseNumber", unique=True)

    cursor = db.enterprise.find(
        {"JuridicalForm": {"$nin": excluded_forms}},
        {"_id": 0, "EnterpriseNumber": 1, "JuridicalForm": 1},
    )
    for enterprise in cursor:
        db.scraping_tracking.update_one(
            {"enterpriseNumber": enterprise["EnterpriseNumber"]},
            {
                "$setOnInsert": {
                    "enterpriseNumber": enterprise["EnterpriseNumber"],
                    "juridicalForm": enterprise["JuridicalForm"],
                    "status": "pending",
                    "lastScrapedAt": None,
                    "documentsDownloaded": 0,
                }
            },
            upsert=True,
        )


def run_scraping_batch() -> None:
    """Traite toutes les entreprises pas encore 'done' de scraping_tracking
    (le filtre sur la forme juridique exclue a déjà été appliqué à la création
    des documents dans init_scraping_tracking)."""
    to_process = db.scraping_tracking.find({"status": {"$ne": "done"}})

    for tracked in to_process:
        raw_number = tracked["enterpriseNumber"].replace(".", "")
        try:
            downloaded = scrape_enterprise(raw_number)
            db.scraping_tracking.update_one(
                {"enterpriseNumber": tracked["enterpriseNumber"]},
                {"$set": {
                    "status": "done",
                    "lastScrapedAt": datetime.now(timezone.utc),
                    "documentsDownloaded": downloaded,
                }},
            )
        except Exception as exc:
            db.scraping_tracking.update_one(
                {"enterpriseNumber": tracked["enterpriseNumber"]},
                {"$set": {"status": "error", "lastScrapedAt": datetime.now(timezone.utc)}},
            )
            print(f"Erreur sur {tracked['enterpriseNumber']}: {exc}")


init_scraping_tracking(EXCLUDED_JURIDICAL_FORMS)
print(f"À traiter : {db.scraping_tracking.count_documents({'status': 'pending'}):,}")
